# PPO + Dirichlet Portfolio Policy — Smoke Test

This notebook verifies the end-to-end PPO pipeline with the custom Dirichlet policy. It deliberately loads the pre-scaled **training** dataset (`environment_data_scaled.pkl`) and uses `PortfolioFeatureExtractor` as PPO's feature extractor. PPO automatically uses CUDA when a CUDA-enabled PyTorch installation and a compatible GPU are available; otherwise it falls back to CPU.

The test checks: 
* actions are valid long-only portfolio weights (non-negative and summing to one).
* trains PPO briefly.
* runs a deterministic evaluation rollout.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from stable_baselines3 import PPO

# Supports execution from either the repository root or notebooks/ directory.
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (cwd, cwd.parent) if (path / 'src').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate the project root containing src/.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, TRAIN_END, get_training_device
from src.custom_policy import PortfolioPolicy
from src.environment import PortfolioEnv
from src.feature_extractor import PortfolioFeatureExtractor

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = get_training_device()

if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')
    print(f'Using GPU: {torch.cuda.get_device_name(DEVICE)}')
else:
    print('CUDA is unavailable; using CPU. Install a CUDA-enabled PyTorch build to train on GPU.')

CUDA is unavailable; using CPU. Install a CUDA-enabled PyTorch build to train on GPU.


In [3]:
scaled_data_path = PROCESSED_DATA_DIR / 'environment_data_scaled.pkl'
raw_price_data_path = PROCESSED_DATA_DIR / 'environment_data.pkl'
if not scaled_data_path.exists():
    raise FileNotFoundError(
        f'{scaled_data_path} was not found. Run notebooks/feature_scaler.ipynb first.'
    )
if not raw_price_data_path.exists():
    raise FileNotFoundError(
        f'{raw_price_data_path} was not found. Raw prices are required for reward calculation.'
    )

# This file contains the scaled training-state dataset. No fitting or re-scaling
# occurs here, preventing a second scaler fit during this PPO test.
train_data = pd.read_pickle(scaled_data_path).copy()
train_data['Date'] = pd.to_datetime(train_data['Date'])
raw_price_data = pd.read_pickle(raw_price_data_path).copy()
raw_price_data['Date'] = pd.to_datetime(raw_price_data['Date'])

assert not train_data.empty
assert train_data['Date'].max() <= pd.Timestamp(TRAIN_END), (
    'The PPO smoke test must use training data only.'
)
assert train_data[['Date', 'Portfolio']].equals(raw_price_data[['Date', 'Portfolio']])

print(f'Loaded {len(train_data):,} scaled training states')
print(train_data.groupby('Portfolio')['Date'].agg(['min', 'max', 'count']))

Loaded 9,057 scaled training states
                    min        max  count
Portfolio                                
Aggressive   2011-01-04 2022-12-30   3019
Conservative 2011-01-04 2022-12-30   3019
Moderate     2011-01-04 2022-12-30   3019


In [4]:
PORTFOLIO = 'Aggressive'
env = PortfolioEnv(
    dataset=train_data,
    portfolio=PORTFOLIO,
    price_data=raw_price_data,
)
observation, info = env.reset(seed=SEED)

# Verify that the custom extractor can encode the scaled 2D state before PPO.
extractor = PortfolioFeatureExtractor(env.observation_space, features_dim=128).to(DEVICE)
with torch.no_grad():
    encoded_state = extractor(
        torch.as_tensor(observation, device=DEVICE).unsqueeze(0)
    )

assert encoded_state.shape == (1, 128)
assert encoded_state.device.type == DEVICE.type
print('Observation shape:', observation.shape)
print('Encoded feature shape:', tuple(encoded_state.shape))
print('Action space:', env.action_space)

Observation shape: (16, 10)
Encoded feature shape: (1, 128)
Action space: Box(0.0, 1.0, (10,), float32)


In [5]:
policy_kwargs = {
    # PPO sends every scaled state through this extractor before the actor/critic MLPs.
    'features_extractor_class': PortfolioFeatureExtractor,
    'features_extractor_kwargs': {'features_dim': 128},
    'net_arch': {'pi': [64, 64], 'vf': [64, 64]},
    'min_concentration': 1e-3,
}

model = PPO(
    policy=PortfolioPolicy,
    env=env,
    learning_rate=3e-4,
    n_steps=64,
    batch_size=32,
    n_epochs=2,
    gamma=0.99,
    seed=SEED,
    device=DEVICE,
    policy_kwargs=policy_kwargs,
    verbose=1,
)

assert isinstance(model.policy.features_extractor, PortfolioFeatureExtractor)
assert next(model.policy.parameters()).device.type == DEVICE.type
print(f'PPO model device: {model.device}')
print(type(model.policy.action_dist).__name__)
print(type(model.policy.features_extractor).__name__)

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
PPO model device: cpu
PortfolioDistribution
PortfolioFeatureExtractor


In [6]:
def assert_simplex_action(action: np.ndarray) -> None:
    assert action.shape == env.action_space.shape
    assert np.all(action >= -1e-7), f'Negative weight found: {action}'
    assert np.all(action <= 1.0 + 1e-7), f'Weight above one found: {action}'
    assert np.isclose(action.sum(), 1.0, atol=1e-5), f'Weights sum to {action.sum()}'

stochastic_action, _ = model.predict(observation, deterministic=False)
deterministic_action, _ = model.predict(observation, deterministic=True)
assert_simplex_action(stochastic_action)
assert_simplex_action(deterministic_action)

print('Stochastic action:', np.round(stochastic_action, 4), 'sum =', stochastic_action.sum())
print('Deterministic action:', np.round(deterministic_action, 4), 'sum =', deterministic_action.sum())

Stochastic action: [0.0246 0.0049 0.0197 0.136  0.1248 0.0804 0.1273 0.3124 0.0182 0.1515] sum = 1.0
Deterministic action: [0.0997 0.0995 0.1008 0.1006 0.1001 0.0995 0.0999 0.1003 0.0997 0.0997] sum = 1.0


In [7]:
# A short run is sufficient for an integration test; increase this for real training.
model.learn(total_timesteps=100_000, progress_bar=True)

f:\SBU\RL\DRL_Final_Project\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter
support
  warnings.warn('install "ipywidgets" for Jupyter support')

----------------------------
| time/              |     |
|    fps             | 520 |
|    iterations      | 1   |
|    time_elapsed    | 0   |
|    total_timesteps | 64  |
----------------------------
-------------------------------------------
| time/                   |               |
|    fps                  | 507           |
|    iterations           | 2             |
|    time_elapsed         | 0             |
|    total_timesteps      | 128           |
| train/                  |               |
|    approx_kl            | 0.00092563685 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | 13.3          |
|    explained_variance   | 0.216         |
|    learning_rate        | 0.0003        |
|    loss                 | -0.0139       |
|    n_updates            | 2             |
|    policy_gradient_loss | -0.00569      |
|    value_loss           | 0.0116        |
-------------------------------------------
-----

In [12]:
evaluation_env = PortfolioEnv(
    dataset=train_data,
    portfolio=PORTFOLIO,
    price_data=raw_price_data,
)
observation, info = evaluation_env.reset(seed=SEED)
episode_reward = 0.0
steps = 0

while steps < 365:
    action, _ = model.predict(observation, deterministic=True)
    assert_simplex_action(action)
    observation, reward, terminated, truncated, info = evaluation_env.step(action)
    episode_reward += reward
    steps += 1
    if terminated or truncated:
        break

print(f'Evaluation steps: {steps}')
print(f'Cumulative reward: {episode_reward:.6f}')
print(f'Final portfolio value: {info["portfolio_value"]:.6f}')
print('PPO + Dirichlet + PortfolioFeatureExtractor smoke test passed.')

Evaluation steps: 365
Cumulative reward: 0.266006
Final portfolio value: 1.253452
PPO + Dirichlet + PortfolioFeatureExtractor smoke test passed.
